In [38]:
import pandas as pd

df = pd.read_csv("wildfires_small.csv")

df.head()

,FIRE_YEAR,DISCOVERY_DOY,DISCOVERY_TIME,LATITUDE,LONGITUDE,STATE,NWCG_CAUSE_CLASSIFICATION,NWCG_GENERAL_CAUSE,OWNER_DESCR,FIRE_SIZE,FIRE_SIZE_CLASS,DISCOVERY_DATE,CONT_DATE
0,1999,106,NaN,44.100000,-69.366667,ME,Missing data/not specified/undetermined,Missing data/not specified/undetermined,STATE,0.20,A,4/16/1999,4/16/1999
1,1996,198,1130.0,48.267500,-105.708300,MT,Human,Equipment and vehicle use,BIA,169.50,D,7/16/1996,7/16/1996
2,1996,256,1355.0,45.733333,-115.615000,ID,Natural,Natural,USFS,0.10,A,9/12/1996,9/12/1996
3,2008,357,NaN,21.396166,-157.984344,HI,Missing data/not specified/undetermined,Missing data/not specified/undetermined,MISSING/NOT SPECIFIED,0.25,A,12/22/2008,NaN
4,2005,328,NaN,33.241062,-84.048659,GA,Human,Missing data/not specified/undetermined,OTHER FEDERAL,47.87,C,11/24/2005,NaN


In [39]:
# Basic dataset information
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum().sort_values(ascending=False))

print("\nDuplicate rows:", df.duplicated().sum())

Shape: (100000, 13)

Columns:
['FIRE_YEAR', 'DISCOVERY_DOY', 'DISCOVERY_TIME', 'LATITUDE', 'LONGITUDE', 'STATE', 'NWCG_CAUSE_CLASSIFICATION', 'NWCG_GENERAL_CAUSE', 'OWNER_DESCR', 'FIRE_SIZE', 'FIRE_SIZE_CLASS', 'DISCOVERY_DATE', 'CONT_DATE']

Missing values:
CONT_DATE                    38618
DISCOVERY_TIME               34115
FIRE_YEAR                        0
DISCOVERY_DOY                    0
LATITUDE                         0
LONGITUDE                        0
STATE                            0
NWCG_CAUSE_CLASSIFICATION        0
NWCG_GENERAL_CAUSE               0
OWNER_DESCR                      0
FIRE_SIZE                        0
FIRE_SIZE_CLASS                  0
DISCOVERY_DATE                   0
dtype: int64

Duplicate rows: 13


In [40]:
# Examine wildfire sizes
print(df["FIRE_SIZE"].describe())

print("\nFire size classes:")
print(df["FIRE_SIZE_CLASS"].value_counts().sort_index())

print("\nInvalid fire sizes:")
print((df["FIRE_SIZE"] <= 0).sum())

count    100000.000000
mean         87.769252
std        2902.091510
min           0.000100
25%           0.100000
50%           0.810000
75%           3.000000
max      537627.000000
Name: FIRE_SIZE, dtype: float64

Fire size classes:
FIRE_SIZE_CLASS
A    37866
B    48059
C    11198
D     1441
E      758
F      454
G      224
Name: count, dtype: int64

Invalid fire sizes:
0


In [41]:
#categorial size function 
def size_category(size):
    if size < 10:
        return "Small"
    elif size < 100:
        return "Medium"
    else:
        return "Large"

df["SIZE_CATEGORY"] = df["FIRE_SIZE"].apply(size_category)

print(df["SIZE_CATEGORY"].value_counts())

print("\nPercentages:")
print(df["SIZE_CATEGORY"].value_counts(normalize=True) * 100)

SIZE_CATEGORY
Small     85925
Medium    11198
Large      2877
Name: count, dtype: int64

Percentages:
SIZE_CATEGORY
Small     85.925
Medium    11.198
Large      2.877
Name: proportion, dtype: float64


In [42]:
from sklearn.model_selection import train_test_split

feature = [
    "FIRE_YEAR",
    "DISCOVERY_DOY",
    "LATITUDE",
    "LONGITUDE",
    "STATE",
    "NWCG_CAUSE_CLASSIFICATION",
    "NWCG_GENERAL_CAUSE"
]

X = df[feature]
y = df["SIZE_CATEGORY"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size = 0.2,
    random_state = 42,
    stratify = y
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)
print("\nTraining Categories:")
print(y_train.value_counts())

Training set: (80000, 7)
Testing set: (20000, 7)

Training Categories:
SIZE_CATEGORY
Small     68740
Medium     8958
Large      2302
Name: count, dtype: int64


In [43]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_features = [
    "FIRE_YEAR",
    "DISCOVERY_DOY",
    "LATITUDE",
    "LONGITUDE"
]

categorical_features = [
    "STATE",
    "NWCG_CAUSE_CLASSIFICATION",
    "NWCG_GENERAL_CAUSE"
]

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(handle_unknown = "ignore"), categorical_features)
])

In [44]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

log_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight={"Small": 1, "Medium": 3, "Large": 10},
        C=0.01,
        random_state=42
    ))
])
log_model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['FIRE_YEAR', 'DISCOVERY_DOY',
                                                   'LATITUDE', 'LONGITUDE']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['STATE',
                                                   'NWCG_CAUSE_CLASSIFICATION',
                                                   'NWCG_GENERAL_CAUSE'])])),
                ('classifier',
                 LogisticRegression(C=0.01,
                                    class_weight={'Large': 10, 'Medium': 3,
                                                  'Small': 1},
                                    max_iter=1000, random_state=42))])

In [45]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred = log_model.predict(X_test)

print("Accuracy", accuracy_score(y_test, y_pred))
print("\nClassification Report")
print(classification_report(y_test, y_pred))
print("\nCConfusion Matrix")
print(confusion_matrix(y_test, y_pred, labels = ["Small", "Medium", "Large"]))

Accuracy 0.79775

Classification Report
              precision    recall  f1-score   support

       Large       0.10      0.18      0.13       575
      Medium       0.31      0.21      0.25      2240
       Small       0.88      0.89      0.89     17185

    accuracy                           0.80     20000
   macro avg       0.43      0.43      0.42     20000
weighted avg       0.80      0.80      0.80     20000


CConfusion Matrix
[[15376   973   836]
 [ 1631   473   136]
 [  406    63   106]]


In [46]:
from sklearn.ensemble import RandomForestClassifier

rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        max_depth=20,
        min_samples_leaf=2,
        class_weight={"Small": 1, "Medium": 5, "Large": 15},
        random_state=42,
        n_jobs=-1
    ))
])
rf_model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['FIRE_YEAR', 'DISCOVERY_DOY',
                                                   'LATITUDE', 'LONGITUDE']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['STATE',
                                                   'NWCG_CAUSE_CLASSIFICATION',
                                                   'NWCG_GENERAL_CAUSE'])])),
                ('classifier',
                 RandomForestClassifier(class_weight={'Large': 15, 'Medium': 5,
                                                      'Small': 1},
                                        max_depth=20, min_samples_leaf=2,
                                        n_estimators=200, n_jobs=-1,
                                        random_state=42))])

In [47]:
rf_pred = rf_model.predict(X_test)

print("Accuracy", accuracy_score(y_test, rf_pred))
print("\nClassification Report")
print(classification_report(y_test, rf_pred))
print("\nCConfusion Matrix")
print(confusion_matrix(y_test, rf_pred, labels = ["Small", "Medium", "Large"]))

Accuracy 0.81475

Classification Report
              precision    recall  f1-score   support

       Large       0.23      0.16      0.19       575
      Medium       0.32      0.34      0.33      2240
       Small       0.90      0.90      0.90     17185

    accuracy                           0.81     20000
   macro avg       0.48      0.47      0.47     20000
weighted avg       0.81      0.81      0.81     20000


CConfusion Matrix
[[15444  1529   212]
 [ 1392   760    88]
 [  362   122    91]]


In [48]:
from sklearn.svm import LinearSVC

svm_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LinearSVC(
        class_weight = "balanced",
        max_iter = 1000,
        random_state = 42
    ))
])
svm_model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['FIRE_YEAR', 'DISCOVERY_DOY',
                                                   'LATITUDE', 'LONGITUDE']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['STATE',
                                                   'NWCG_CAUSE_CLASSIFICATION',
                                                   'NWCG_GENERAL_CAUSE'])])),
                ('classifier',
                 LinearSVC(class_weight='balanced', random_state=42))])

In [49]:
svm_pred = svm_model.predict(X_test)

print("Accuracy", accuracy_score(y_test, svm_pred))
print("\nClassification Report")
print(classification_report(y_test, svm_pred))
print("\nCConfusion Matrix")
print(confusion_matrix(y_test, svm_pred, labels = ["Small", "Medium", "Large"]))

Accuracy 0.8234

Classification Report
              precision    recall  f1-score   support

       Large       0.13      0.08      0.10       575
      Medium       0.31      0.23      0.26      2240
       Small       0.88      0.93      0.90     17185

    accuracy                           0.82     20000
   macro avg       0.44      0.41      0.42     20000
weighted avg       0.80      0.82      0.81     20000


CConfusion Matrix
[[15916  1044   225]
 [ 1664   507    69]
 [  460    70    45]]


In [50]:
X_train_final, X_val, y_train_final, y_val = train_test_split(
    X_train,
    y_train,
    test_size = 0.2,
    random_state = 42,
    stratify = y_train
)

print("Training set:", X_train_final.shape)
print("Validation:", X_val.shape)

Training set: (64000, 7)
Validation: (16000, 7)


In [51]:
from sklearn.metrics import f1_score
from sklearn.base import clone

for depth in [10, 15, 20, None]:
    model = clone(rf_model)
    model.set_params(classifier__max_depth = depth)

    model.fit(X_train_final, y_train_final)
    predictions = model.predict(X_val)

    score = f1_score(y_val, predictions, average = "macro")

    print("Max Depth", depth, "F1", round(score, 4))

Max Depth 10 F1 0.457
Max Depth 15 F1 0.4803
Max Depth 20 F1 0.4829
Max Depth None F1 0.458


In [52]:
for leaf in [1, 2, 5, 10]:
    model = clone(rf_model)

    model.set_params(
        classifier__max_depth = 20,
        classifier__min_samples_leaf = leaf
    )

    model.fit(X_train_final, y_train_final)
    predictions = model.predict(X_val)

    score = f1_score(y_val, predictions, average="macro")

    print("Min Samples Leaf:", leaf, "Macro F1:", round(score, 4))

Min Samples Leaf: 1 Macro F1: 0.4573
Min Samples Leaf: 2 Macro F1: 0.4829
Min Samples Leaf: 5 Macro F1: 0.4904
Min Samples Leaf: 10 Macro F1: 0.4859


In [53]:
for trees in [50, 100, 200, 300]:
    model = clone(rf_model)

    model.set_params(
        classifier__max_depth = 20,
        classifier__min_samples_leaf = 2,
        classifier__n_estimators = trees
    )

    model.fit(X_train_final, y_train_final)
    predictions = model.predict(X_val)

    score = f1_score(y_val, predictions, average="macro")

    print("Trees:", trees, "Macro F1:", round(score, 4))

Trees: 50 Macro F1: 0.486
Trees: 100 Macro F1: 0.4812
Trees: 200 Macro F1: 0.4829
Trees: 300 Macro F1: 0.4828


In [54]:

best_rf = clone(rf_model)

best_rf.fit(X_train_final, y_train_final)

best_pred = best_rf.predict(X_val)

print("Validation Accuracy:", accuracy_score(y_val, best_pred))

print("\nClassification Report:")
print(classification_report(y_val, best_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(
    y_val,
    best_pred,
    labels=["Small", "Medium", "Large"]
))

Validation Accuracy: 0.8199375

Classification Report:
              precision    recall  f1-score   support

       Large       0.30      0.18      0.22       460
      Medium       0.32      0.33      0.32      1792
       Small       0.90      0.91      0.90     13748

    accuracy                           0.82     16000
   macro avg       0.51      0.47      0.48     16000
weighted avg       0.82      0.82      0.82     16000


Confusion Matrix:
[[12445  1171   132]
 [ 1145   593    54]
 [  285    94    81]]


In [55]:
models = {
    "Logistic Regression": clone(log_model),
    "Random Forest": clone(best_rf),
    "Linear SVM": clone(svm_model)
}

for name, model in models.items():
    model.fit(X_train_final, y_train_final)
    predictions = model.predict(X_val)

    accuracy = accuracy_score(y_val, predictions)
    f1 = f1_score(y_val, predictions, average="macro")

    print(name)
    print("Accuracy:", round(accuracy, 4))
    print("Macro F1:", round(f1, 4))
    print()

Logistic Regression
Accuracy: 0.8113
Macro F1: 0.4337

Random Forest
Accuracy: 0.8199
Macro F1: 0.4829

Linear SVM
Accuracy: 0.8254
Macro F1: 0.428

